# 0. Install and import the required dependencies

**You may add or remove based on your assigned model!**

In [ ]:
!pip install -U tokenizers transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# 1. Authentication & Model selection

**Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

**CHANGE THIS TO YOUR ASSIGNED MODEL**

In [ ]:
# A smaller model that fits on a free Colab GPU
model_name = "ByteDance-Seed/Seed-Coder-8B-Instruct"

# 2. Hardware optimization (Quantization)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # What is loaded in 4 bit? why?
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 3. Load the tokenizer & model

In [ ]:
print(f"Loading Tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=False # Does your model need it?
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # What about right?

Loading Tokenizer for ByteDance-Seed/Seed-Coder-8B-Instruct...


config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

In [ ]:
print("Loading Model on Colab T4 GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True, # Does your model need it?
    quantization_config=bnb_config, # from section 2 above
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading Model on Colab T4 GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

# 4. INFERENCE & HYPERPARAMETER TUNING

**Design the prompt (Does this design/technique have a name?)**

In [ ]:
prompt = [
    {
        "role": "system",
        "content": "You are a helpful software assistant. Your job is to explain the functionality of the provided code in simple terms."
    },
    {
        "role": "user",
        "content": """Please analyze the following Java source code.

Provide:

1. A brief summary of the program.
2. Step-by-step explanation.
3. Explanation of all variables.
4. Explanation of the output.
5. Time complexity.
6. Potential improvements.

Use simple language suitable for a beginner programmer.

        <source_code>
        public class Task {

            public static void main(String[] args) {
                String id1   = "AMQ-2104";
                double files = 8.0;
                String id2   = "AMQ-317";
                double dmm   = 0.45;

                int n = Integer.parseInt(id1.split("-")[1])
                      + Integer.parseInt(id2.split("-")[1]);

                int digits = 0;
                while (n > 0) {
                    n = n / 10;
                    digits++;
                }

                int impact = (int) (files * dmm);

                System.out.println("Combined digits: " + digits);
                System.out.println("Impact: " + impact);
            }
        }
        </source_code>"""
    }
]

**Convert the prompt into tokens**

In [ ]:
inputs = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

**Understand these hyperparameters and then tune them for better inference**

In [ ]:
print("Generating response...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350, # which tokens?
    temperature=0.3, # ?
    top_p=0.9, # ?
    repetition_penalty=1.1,
    do_sample=True, # What if this toggle is false?
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

Generating response...



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


**Convert the output back from tokens into human-readable text and display it**

In [ ]:
input_length = inputs['input_ids'].shape[1]
response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

print("Model output:")
print(response)

Model output:
Here's an analysis of the provided Java code:

1. Brief summary:
This program calculates and displays information based on two IDs, file sizes, and a multiplier value.

2. Step-by-step explanation:
a. Set up initial values for two IDs and their corresponding file sizes.
b. Extract numbers from the IDs, add them together, and count the number of digits in the sum.
c. Calculate an 'impact' value by multiplying file sizes with each other.
d. Display the results.

3. Variables explained:
- id1, id2: Strings representing task or project identifiers.
- files, dmm: Double values representing file sizes or multipliers.
- n: Integer used to store the sum of extracted numbers from IDs.
- digits: Integer to count the number of digits in n.
- impact: Integer calculated as the product of files and dmm.

4. Output explanation:
The program will print two lines:
- The first line shows how many digits are in the sum of numbers from both IDs.
- The second line displays the calculated impac